# UTKFace × Age Gate Lab（age-gate/1.0.0）

**18〜75歳の範囲内か範囲外かだけを判定するゲートモデル**を学習する。年齢そのものは推定しない。

`utkface_symmetry_lab.ipynb`（symmetry-3d/1.0.0）と同じ幾何計算・同じ姿勢ゲート・同じ
データセット注釈を使い、そこへ「年齢帯の手がかりになる顔の比率」を足して学習する。

## なぜ作るのか

アプリの `FR-022` は「対象範囲外には推論を提供しない」と定めているが、現在の
`age-gender-recognition-retail-0013` は 18〜75 の範囲でしか学習されておらず、
**範囲外だと言う手段を持たない**。子どもを入力しても 20 代の数字を返す。
このゲートはその穴だけを埋める。

## 用途の限定（必ず守る）

- 用途は**「参考年齢を表示してよいかどうか」の判定のみ**。
- **年齢認証・本人確認・未成年判定・入場制限には使わない。**
- 出力は断定ではなく、閾値つきのゲート。迷ったら「範囲外」に倒す（参考年齢を出さない側が安全側）。
- UTKFace は**非商用研究限定**。年齢注釈自体の誤りは検証していない。
- 群（人種・性別）ごとの誤り率を必ず出す。偏りを隠したまま配備しない。

## 出力（`/content`）

| ファイル | 内容 |
|---|---|
| `age_gate_model.onnx` | 学習済みゲートモデル（アプリへ渡す唯一の成果物） |
| `age_gate_feature_spec.json` | 特徴量の順序と定義。アプリ側はこの順で組み立てる |
| `age_gate_model_card.md` | モデルカード（用途・限界・評価） |
| `utkface_agegate_eval.json` | 固定テストセットの評価レポート |
| `utkface_agegate_summary.json` | 実行条件・件数・版・ハッシュ・警告 |
| `utkface_agegate_records.csv` | 個票（年齢ラベル・姿勢・比率特徴・予測確率） |

画像とランドマーク座標は保存しない。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# -*- coding: utf-8 -*-
ACCEPT_UTKFACE_NONCOMMERCIAL_RESEARCH_TERMS = True
assert ACCEPT_UTKFACE_NONCOMMERCIAL_RESEARCH_TERMS, (
    'UTKFace公式ページの非商用研究限定条件と所属機関のルールを確認してください。'
)

import subprocess
import sys

PACKAGES = [
    'kagglehub==1.0.2',
    'opencv-contrib-python==5.0.0.93',  # 不変条件9: contrib 1本に限定（mediapipe の依存）
    'mediapipe==0.10.35',
    'pillow==12.3.0',
    # --- ここから学習用。symmetry notebook には無い ---
    'lightgbm==4.7.0',
    'onnx==1.22.0',
    'onnxmltools==1.16.0',
    'onnxruntime==1.25.0',
]

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
subprocess.run(
    [sys.executable, '-m', 'uv', 'pip', 'uninstall', '--system',
     'opencv-python', 'opencv-python-headless',
     'opencv-contrib-python', 'opencv-contrib-python-headless'],
    check=False, capture_output=True, text=True,
)
subprocess.run(
    [sys.executable, '-m', 'uv', 'pip', 'install', '--system', '--quiet', *PACKAGES],
    check=True,
)
print('依存関係を uv で準備しました。')

In [ ]:
import hashlib
from pathlib import Path
from urllib.request import urlopen

MODEL_VERSION = 'mediapipe-face-landmarker/float16-1'
MODEL_URL = (
    'https://storage.googleapis.com/mediapipe-models/'
    'face_landmarker/face_landmarker/float16/1/face_landmarker.task'
)
MODEL_SIZE_BYTES = 3_758_596
MODEL_SHA256 = (
    '64184e229b263107bc2b804c6625db134'
    '1ff2bb731874b0bcc2fe6544e0bc9ff'
)
MODEL_DIR = Path('/content/face-symmetry-models')
MODEL_PATH = MODEL_DIR / 'face_landmarker.task'
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def model_is_valid(path: Path) -> bool:
    return (
        path.is_file()
        and path.stat().st_size == MODEL_SIZE_BYTES
        and sha256_file(path) == MODEL_SHA256
    )


if not model_is_valid(MODEL_PATH):
    temporary_path = MODEL_PATH.with_suffix('.task.part')
    digest = hashlib.sha256()
    received = 0
    with urlopen(MODEL_URL, timeout=60) as response:
        with temporary_path.open('wb') as handle:
            while chunk := response.read(1024 * 1024):
                received += len(chunk)
                digest.update(chunk)
                handle.write(chunk)
    assert received == MODEL_SIZE_BYTES, f'model size mismatch: {received}'
    assert digest.hexdigest() == MODEL_SHA256, 'model SHA-256 mismatch'
    temporary_path.replace(MODEL_PATH)

assert model_is_valid(MODEL_PATH), '固定モデルを検証できませんでした。'
print(f'landmark model: {MODEL_VERSION}')

# 年齢・性別の推論モデルは使わない。正解ラベルは UTKFace のファイル名注釈。

In [ ]:
from collections import Counter

import kagglehub

UTKFACE_OFFICIAL_PAGE_URL = 'https://susanqq.github.io/UTKFace/'
KAGGLE_MIRRORS = (
    'moritzm00/utkface-cropped/versions/1',
    'jangedoo/utkface-new/versions/1',
)
download_errors = []
dataset_root = None
DATASET_MIRROR_HANDLE = None

for mirror_handle in KAGGLE_MIRRORS:
    try:
        dataset_root = Path(kagglehub.dataset_download(mirror_handle))
        DATASET_MIRROR_HANDLE = mirror_handle
        break
    except Exception as error:
        download_errors.append(f'{mirror_handle}: {type(error).__name__}: {error}')

if dataset_root is None or DATASET_MIRROR_HANDLE is None:
    raise RuntimeError(
        '公開KaggleミラーからUTKFaceを取得できませんでした。'
        'Kaggle側が認証を要求する場合はColabのSecretsへKAGGLE_API_TOKENを登録してください。\n'
        + '\n'.join(download_errors)
    )

all_images = [
    path for path in dataset_root.rglob('*')
    if path.is_file() and path.suffix.lower() in {'.jpg', '.jpeg'}
]
images_per_directory = Counter(path.parent for path in all_images)
assert images_per_directory, '取得データ内にJPEG画像がありません。'
DATASET_DIR, _ = images_per_directory.most_common(1)[0]
image_paths = sorted(path for path in all_images if path.parent == DATASET_DIR)
assert 20_000 <= len(image_paths) <= 25_000, (
    f'UTKFace Aligned & Cropped本体を特定できませんでした: {len(image_paths):,}枚'
)

inventory_digest = hashlib.sha256()
for path in image_paths:
    inventory_digest.update(f'{path.name}\0{path.stat().st_size}\n'.encode('utf-8'))
DATASET_INVENTORY_SHA256 = inventory_digest.hexdigest()
DATASET_MIRROR_URL = (
    'https://www.kaggle.com/datasets/' + DATASET_MIRROR_HANDLE.split('/versions/')[0]
)

print(f'mirror: {DATASET_MIRROR_HANDLE}')
print(f'inventory SHA-256: {DATASET_INVENTORY_SHA256}')
print(f'images: {len(image_paths):,}')

In [ ]:
import os

# ---- ゲートの定義 ----
IN_RANGE_MIN_AGE = 18      # アプリ FR-022 の対象下限
IN_RANGE_MAX_AGE = 75      # 同上限

# ---- 姿勢ゲート：アプリの厳格ゲート内側と一致させる ----
# 配備時にこのゲートを通ったフレームにしかモデルを当てないので、学習も同じ条件で行う。
MAX_ABS_YAW_DEG = 15.0
MAX_ABS_PITCH_DEG = 15.0
MAX_ABS_ROLL_DEG = 25.0

# ---- 年齢ゲートは掛けない（範囲外の画像こそが負例）----
MAX_IMAGES = None          # 動作確認だけなら 300
RANDOM_SEED = 20260810
PROGRESS_EVERY = 500

USE_MULTIPROCESSING = True
N_WORKERS = min(os.cpu_count() or 2, 4)
CHUNK = 32

# ---- 学習の条件 ----
SPLIT_RATIOS = (0.60, 0.20, 0.20)    # train / validation / test
INCLUDE_LANDMARK_COORDS = True       # 「手作り比率」vs「正規化座標そのまま」の比較を行う
SAVE_LANDMARK_MATRIX = False         # ランドマーク座標をディスクへ保存しない（原本と同じ姿勢）
EXPORT_FEATURE_SET = 'ratio'         # ONNX へ書き出す方（'ratio' か 'landmark'）
TARGET_UNDER18_RECALLS = (0.90, 0.95, 0.99)   # 動作点の候補
PRIMARY_TARGET_UNDER18_RECALL = 0.95          # 既定で採用する動作点
MIN_GROUP_FOR_REPORT = 30            # これ未満の群は誤り率を出さない（数字が独り歩きするため）

print({
    'in_range': [IN_RANGE_MIN_AGE, IN_RANGE_MAX_AGE],
    'pose_gate': [MAX_ABS_YAW_DEG, MAX_ABS_PITCH_DEG, MAX_ABS_ROLL_DEG],
    'workers': N_WORKERS,
    'label_source': 'dataset filename annotation (推定しない)',
})

In [ ]:
%%writefile /content/fsl_age_gate.py
"""18〜75歳の範囲内/範囲外を判定するゲートモデル用の特徴抽出ワーカ。

`utkface_symmetry_lab.ipynb`（symmetry-3d/1.0.0）の幾何計算をそのまま流用し、
そこへ「年齢帯の手がかりになる顔の比率」を足したものを 1 枚 1 行で返す。

このワーカは年齢を推定しない。UTKFace のファイル名注釈（age_gender_race_datetime）
が持つ実年齢を正解ラベルとして使い、学習は後段のセルで行う。

年齢ゲート（18〜75）はここでは掛けない。範囲外の画像こそが負例だからである。
姿勢ゲートはアプリと同じ値で掛ける（配備条件と同じ条件で学習するため）。
"""

from __future__ import annotations

import math
import re
from pathlib import Path

import numpy as np

# ---- 指標定数：configs/metrics/symmetry-3d-v1.json と同一 ----
METRIC_VERSION = "symmetry-3d/1.0.0"
LANDMARK_MAP_VERSION = "mediapipe-478-mirror/1.0.0"
FEATURE_SPEC_VERSION = "age-gate-features/1.0.0"
LANDMARK_COUNT = 478
PART_NAMES = ("contour", "eyes", "nose", "mouth")
LEFT_EYE_CORNERS = (33, 133)
RIGHT_EYE_CORNERS = (263, 362)
PART_PAIRS = {
    "contour": (
        (21, 251), (54, 284), (58, 288), (67, 297), (93, 323),
        (103, 332), (109, 338), (127, 356), (132, 361), (136, 365),
        (148, 377), (149, 378), (150, 379), (162, 389), (172, 397),
        (176, 400), (234, 454),
    ),
    "eyes": (
        (7, 249), (33, 263), (46, 276), (52, 282), (53, 283),
        (55, 285), (63, 293), (65, 295), (66, 296), (70, 300),
        (105, 334), (107, 336), (133, 362), (144, 373),
        (145, 374), (153, 380), (154, 381), (155, 382),
        (157, 384), (158, 385), (159, 386), (160, 387),
        (161, 388), (163, 390), (173, 398), (246, 466),
    ),
    "nose": (
        (45, 275), (48, 278), (64, 294), (97, 326), (98, 327),
        (115, 344), (220, 440),
    ),
    "mouth": (
        (37, 267), (39, 269), (74, 270), (76, 291), (78, 306),
        (81, 311), (82, 312), (84, 314), (87, 317), (88, 319),
        (90, 321), (178, 403), (181, 405),
    ),
}
POOLED_PAIRS = tuple(pair for name in PART_NAMES for pair in PART_PAIRS[name])
SERIES = (*PART_NAMES, "overall")

DISTANCE_AT_FULL_SCORE = 0.010
DISTANCE_AT_ZERO_SCORE = 0.050
MIN_EYE_DISTANCE = 1e-9

UTKFACE_NAME_PATTERN = re.compile(r"^(\d{1,3})_([01])_([0-4])_\d+")
GENDER_LABELS = {0: "male", 1: "female"}
RACE_LABELS = {0: "White", 1: "Black", 2: "Asian", 3: "Indian", 4: "Others"}

# ---- 事前計算 ----
_USED = sorted({index for pair in POOLED_PAIRS for index in pair})
_USED_ARR = np.asarray(_USED, dtype=np.int64)
_POSITION = {landmark: order for order, landmark in enumerate(_USED)}
_SWAP = np.arange(len(_USED))
for _left, _right in POOLED_PAIRS:
    _SWAP[_POSITION[_left]], _SWAP[_POSITION[_right]] = _POSITION[_right], _POSITION[_left]
_PART_ROWS = {
    name: np.asarray(
        [_POSITION[index] for pair in PART_PAIRS[name] for index in pair], dtype=np.int64
    )
    for name in PART_NAMES
}
_LEFT2 = list(LEFT_EYE_CORNERS)
_RIGHT2 = list(RIGHT_EYE_CORNERS)

# ---- 比率特徴で使うランドマーク（MediaPipe FaceMesh の標準 index）----
IDX = {
    "forehead": 10,      # 額の正中最上部
    "chin": 152,         # 顎の正中最下部
    "cheek_l": 234,      # 左頬の最外
    "cheek_r": 454,      # 右頬の最外
    "jaw_l": 58,         # 左下顎角
    "jaw_r": 288,        # 右下顎角
    "nasion": 168,       # 両眼のあいだ（鼻根）
    "nose_tip": 1,       # 鼻尖
    "subnasale": 2,      # 鼻下
    "nose_wing_l": 98,
    "nose_wing_r": 327,
    "mouth_l": 61,
    "mouth_r": 291,
    "lip_top": 0,        # 上唇の外側正中
    "lip_bottom": 17,    # 下唇の外側正中
    "eye_l_out": 33, "eye_l_in": 133, "eye_l_top": 159, "eye_l_bottom": 145,
    "eye_r_out": 263, "eye_r_in": 362, "eye_r_top": 386, "eye_r_bottom": 374,
    "brow_l": 105, "brow_r": 334,
}

# 特徴量の順序はここで固定する。アプリ側は同じ順序で組み立てる。
GEOMETRY_FEATURE_NAMES = (
    "face_height", "face_width", "face_aspect",
    "forehead_ratio", "midface_ratio", "lower_face_ratio", "chin_ratio",
    "nose_length_ratio", "nose_width_ratio", "nose_depth",
    "mouth_width_ratio", "mouth_position_ratio", "lip_height_ratio",
    "jaw_width_ratio", "cheek_height_ratio", "jaw_taper",
    "eye_width_left", "eye_width_right",
    "eye_open_left", "eye_open_right",
    "eye_aspect_left", "eye_aspect_right",
    "brow_gap_left", "brow_gap_right",
    "eye_to_nose_ratio", "eye_to_mouth_ratio",
    "depth_forehead", "depth_chin", "depth_cheek_mean", "depth_spread",
    "contour_depth_range", "face_convexity",
)
POSE_FEATURE_NAMES = ("yaw_deg", "pitch_deg", "roll_deg")
SYMMETRY_FEATURE_NAMES = (
    "s3d_distance",
    "distance_contour", "distance_eyes", "distance_nose",
    "distance_mouth", "distance_overall",
)
#: 学習に使う特徴量の正式な順序（landmark 座標を足す場合はこの後ろへ連結する）
TABULAR_FEATURE_NAMES = (
    *GEOMETRY_FEATURE_NAMES, *POSE_FEATURE_NAMES, *SYMMETRY_FEATURE_NAMES,
)

# ---- ゲート/モデル（init_worker で設定）----
MODEL_PATH = ""
MAX_ABS_YAW_DEG = 15.0
MAX_ABS_PITCH_DEG = 15.0
MAX_ABS_ROLL_DEG = 25.0
RETURN_LANDMARK_COORDS = True
_LANDMARKER = None


def parse_utkface_name(name: str):
    """`25_0_2_2017...jpg` → 注釈 dict。読めなければ None（推測で埋めない）。"""
    match = UTKFACE_NAME_PATTERN.match(name)
    if match is None:
        return None
    age, gender_code, race_code = (int(group) for group in match.groups())
    return {
        "age": age,
        "gender_code": gender_code,
        "gender": GENDER_LABELS[gender_code],
        "race_code": race_code,
        "race": RACE_LABELS[race_code],
    }


def normalize_landmarks_3d(points: np.ndarray) -> np.ndarray:
    """geometry.normalize_landmarks_3d と同一。スケールは 3D の両眼間距離。

    返り値は「両眼間距離 = 1」「両眼中点 = 原点」「両眼を水平へ回した」座標。
    以降の比率特徴はすべてこの単位で測るので、画像サイズと顔の大きさに依存しない。
    """
    array = np.asarray(points, dtype=np.float64)
    left2 = array[_LEFT2, :2].mean(axis=0)
    right2 = array[_RIGHT2, :2].mean(axis=0)
    delta = right2 - left2
    distance2d = float(np.hypot(delta[0], delta[1]))
    if distance2d <= MIN_EYE_DISTANCE:
        raise ValueError("eye distance too small")
    cosine = float(delta[0]) / distance2d
    sine = -float(delta[1]) / distance2d
    rotation = np.array([[cosine, -sine], [sine, cosine]], dtype=np.float64)
    center = (left2 + right2) / 2.0
    left3 = array[_LEFT2].mean(axis=0)
    right3 = array[_RIGHT2].mean(axis=0)
    scale = float(np.linalg.norm(right3 - left3))
    if scale <= MIN_EYE_DISTANCE:
        raise ValueError("eye distance too small")
    plane = ((array[:, :2] - center) @ rotation.T) / scale
    depth = (array[:, 2] - float(left3[2] + right3[2]) / 2.0) / scale
    return np.column_stack([plane, depth])


def axis_split(residual: np.ndarray) -> dict:
    return {
        "distance": float(np.sqrt((residual**2).sum(axis=1).mean())),
        "plane": float(np.sqrt((residual[:, :2] ** 2).sum(axis=1).mean())),
        "depth": float(np.sqrt((residual[:, 2] ** 2).mean())),
    }


def compute_scores_3d(normalized: np.ndarray) -> dict:
    """正規化済み座標から symmetry-3d/1.0.0 の距離一式を出す。"""
    original = normalized[_USED_ARR]
    mirrored = original.copy()
    mirrored[:, 0] *= -1.0
    mirrored = mirrored[_SWAP]
    centred = original - original.mean(axis=0)
    mirrored_centred = mirrored - mirrored.mean(axis=0)
    left_vectors, _, right_vectors = np.linalg.svd(mirrored_centred.T @ centred)
    rotation = left_vectors @ right_vectors
    if np.linalg.det(rotation) < 0:
        left_vectors[:, -1] *= -1.0
        rotation = left_vectors @ right_vectors
    residual = centred - (mirrored_centred @ rotation)
    result = {name: axis_split(residual[_PART_ROWS[name]]) for name in PART_NAMES}
    result["overall"] = axis_split(residual)
    result["s_3d"] = math.sqrt(
        sum(result[name]["distance"] ** 2 for name in PART_NAMES) / len(PART_NAMES)
    )
    return result


def score_points(distance):
    if distance is None or not math.isfinite(distance):
        return None
    span = DISTANCE_AT_ZERO_SCORE - DISTANCE_AT_FULL_SCORE
    return min(100.0, max(0.0, 100.0 * (DISTANCE_AT_ZERO_SCORE - distance) / span))


def pose_degrees(matrix: np.ndarray):
    """pose.pose_from_matrix と同一の yaw/pitch/roll（度）。"""
    linear = np.asarray(matrix, dtype=np.float64)[:3, :3]
    left, singular_values, right = np.linalg.svd(linear)
    if singular_values.min() <= 1e-7:
        raise ValueError("pose matrix is degenerate")
    rotation = left @ right
    if np.linalg.det(rotation) < 0:
        raise ValueError("pose matrix contains reflection")
    sin_pitch = float(np.clip(-rotation[1, 2], -1.0, 1.0))
    pitch = math.asin(sin_pitch)
    cos_pitch = math.sqrt(max(0.0, 1.0 - sin_pitch * sin_pitch))
    if cos_pitch > 1e-7:
        yaw = math.atan2(float(rotation[0, 2]), float(rotation[2, 2]))
        roll = math.atan2(float(rotation[1, 0]), float(rotation[1, 1]))
    else:
        yaw = math.atan2(-float(rotation[2, 0]), float(rotation[0, 0]))
        roll = 0.0
    return tuple(math.degrees(value) for value in (yaw, pitch, roll))


def geometry_features(normalized: np.ndarray) -> dict:
    """正規化座標から、年齢帯の手がかりになる比率を作る。

    単位はすべて「両眼間距離 = 1」。y は画像座標なので下向きが正、
    原点は両眼中点。したがって額は y<0、顎は y>0 になる。

    子どもと成人で系統的に違うのは主に次の3つで、そこを狙って並べている。
      - 額の相対的な高さ（子どもは頭蓋が大きく額の割合が大きい）
      - 中顔面・下顔面の伸び（成長とともに下顔面が伸びる）
      - 鼻の高さと顎の張り出し（立体感は成長で増える）
    """
    p = normalized
    idx = IDX

    def point(key):
        return p[idx[key]]

    def dist(a, b):
        return float(np.linalg.norm(point(a) - point(b)))

    def dy(a, b):
        return float(abs(point(a)[1] - point(b)[1]))

    def dx(a, b):
        return float(abs(point(a)[0] - point(b)[0]))

    face_height = dy("forehead", "chin")
    face_width = dx("cheek_l", "cheek_r")
    if face_height <= 1e-6 or face_width <= 1e-6:
        raise ValueError("degenerate face box")

    eye_y = 0.0  # 正規化の定義により両眼中点が原点
    forehead_y = float(point("forehead")[1])
    chin_y = float(point("chin")[1])
    nasion_y = float(point("nasion")[1])
    subnasale_y = float(point("subnasale")[1])
    mouth_y = float((point("lip_top")[1] + point("lip_bottom")[1]) / 2.0)

    eye_width_left = dx("eye_l_out", "eye_l_in")
    eye_width_right = dx("eye_r_out", "eye_r_in")
    eye_open_left = dy("eye_l_top", "eye_l_bottom")
    eye_open_right = dy("eye_r_top", "eye_r_bottom")

    contour_rows = _PART_ROWS["contour"]
    contour_depth = p[_USED_ARR][contour_rows][:, 2]

    features = {
        # --- 顔全体の寸法（両眼間距離を 1 とした比） ---
        "face_height": face_height,
        "face_width": face_width,
        "face_aspect": face_height / face_width,
        # --- 縦方向の配分。合計はおおよそ 1 になる ---
        "forehead_ratio": (eye_y - forehead_y) / face_height,
        "midface_ratio": (subnasale_y - nasion_y) / face_height,
        "lower_face_ratio": (chin_y - subnasale_y) / face_height,
        "chin_ratio": (chin_y - float(point("lip_bottom")[1])) / face_height,
        # --- 鼻 ---
        "nose_length_ratio": (float(point("nose_tip")[1]) - nasion_y) / face_height,
        "nose_width_ratio": dx("nose_wing_l", "nose_wing_r") / face_width,
        "nose_depth": float(abs(point("nose_tip")[2])),
        # --- 口 ---
        "mouth_width_ratio": dx("mouth_l", "mouth_r") / face_width,
        "mouth_position_ratio": (mouth_y - eye_y) / face_height,
        "lip_height_ratio": dy("lip_top", "lip_bottom") / face_height,
        # --- 輪郭 ---
        "jaw_width_ratio": dx("jaw_l", "jaw_r") / face_width,
        "cheek_height_ratio": (float(point("cheek_l")[1] + point("cheek_r")[1]) / 2.0
                               - eye_y) / face_height,
        "jaw_taper": dx("jaw_l", "jaw_r") / max(dx("cheek_l", "cheek_r"), 1e-6),
        # --- 目 ---
        "eye_width_left": eye_width_left,
        "eye_width_right": eye_width_right,
        "eye_open_left": eye_open_left,
        "eye_open_right": eye_open_right,
        "eye_aspect_left": eye_open_left / max(eye_width_left, 1e-6),
        "eye_aspect_right": eye_open_right / max(eye_width_right, 1e-6),
        "brow_gap_left": dy("brow_l", "eye_l_top"),
        "brow_gap_right": dy("brow_r", "eye_r_top"),
        # --- 目を基準にした縦位置 ---
        "eye_to_nose_ratio": (float(point("nose_tip")[1]) - eye_y) / face_height,
        "eye_to_mouth_ratio": (mouth_y - eye_y) / face_height,
        # --- 立体感（z は両眼中点を 0、両眼間距離を 1 とした奥行き）---
        "depth_forehead": float(point("forehead")[2]),
        "depth_chin": float(point("chin")[2]),
        "depth_cheek_mean": float((point("cheek_l")[2] + point("cheek_r")[2]) / 2.0),
        "depth_spread": float(p[_USED_ARR][:, 2].std()),
        "contour_depth_range": float(contour_depth.max() - contour_depth.min()),
        "face_convexity": float(abs(point("nose_tip")[2] - point("chin")[2])),
    }
    assert tuple(features) == GEOMETRY_FEATURE_NAMES, "特徴量の順序が仕様とずれています"
    return features


def init_worker(model_path: str, gates: tuple, return_landmark_coords: bool = True):
    global MODEL_PATH, MAX_ABS_YAW_DEG, MAX_ABS_PITCH_DEG, MAX_ABS_ROLL_DEG
    global RETURN_LANDMARK_COORDS
    MODEL_PATH = model_path
    MAX_ABS_YAW_DEG, MAX_ABS_PITCH_DEG, MAX_ABS_ROLL_DEG = gates
    RETURN_LANDMARK_COORDS = return_landmark_coords


def _get_landmarker():
    global _LANDMARKER
    if _LANDMARKER is None:
        from mediapipe.tasks.python import BaseOptions
        from mediapipe.tasks.python.vision import (
            FaceLandmarker, FaceLandmarkerOptions, RunningMode,
        )
        options = FaceLandmarkerOptions(
            base_options=BaseOptions(
                model_asset_path=str(MODEL_PATH),
                delegate=BaseOptions.Delegate.CPU,
            ),
            running_mode=RunningMode.IMAGE,
            num_faces=2,
            output_facial_transformation_matrixes=True,
        )
        _LANDMARKER = FaceLandmarker.create_from_options(options)
    return _LANDMARKER


def analyze_one_path(path_str: str):
    """1 枚を解析して ('ok', 測定値) か ('除外理由', None) を返す。

    年齢での除外は行わない。姿勢ゲートはアプリと同じ値で掛ける。
    """
    import mediapipe as mp
    from PIL import Image, ImageOps

    path = Path(path_str)
    try:
        with Image.open(path) as source:
            image = ImageOps.exif_transpose(source).convert("RGB")
            rgb = np.asarray(image, dtype=np.uint8)
    except Exception:
        return ("image_decode_failed", None)

    landmarker = _get_landmarker()
    try:
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb))
        detection = landmarker.detect(mp_image)
    except Exception:
        return ("landmark_detection_error", None)

    faces = list(detection.face_landmarks or [])
    if len(faces) == 0:
        return ("no_face", None)
    if len(faces) > 1:
        return ("multiple_faces", None)
    if len(faces[0]) != LANDMARK_COUNT:
        return ("unexpected_landmark_count", None)

    matrices = list(detection.facial_transformation_matrixes or [])
    if len(matrices) != 1:
        return ("pose_missing", None)
    try:
        yaw, pitch, roll = pose_degrees(np.asarray(matrices[0]))
    except (ValueError, np.linalg.LinAlgError):
        return ("pose_missing", None)
    if (
        abs(yaw) > MAX_ABS_YAW_DEG
        or abs(pitch) > MAX_ABS_PITCH_DEG
        or abs(roll) > MAX_ABS_ROLL_DEG
    ):
        return ("pose_outside_limits", None)

    height, width = rgb.shape[:2]
    points = np.asarray(
        [(p.x * width, p.y * height, p.z * width) for p in faces[0]], dtype=np.float64
    )
    if not np.isfinite(points).all():
        return ("non_finite_landmarks", None)

    try:
        normalized = normalize_landmarks_3d(points)
        scores = compute_scores_3d(normalized)
        geometry = geometry_features(normalized)
    except (ValueError, np.linalg.LinAlgError, IndexError):
        return ("feature_error", None)

    record = {
        "image_width": int(width),
        "image_height": int(height),
        "yaw_deg": float(yaw),
        "pitch_deg": float(pitch),
        "roll_deg": float(roll),
        "s3d_distance": float(scores["s_3d"]),
        "score_points": score_points(scores["s_3d"]),
        **{f"distance_{name}": float(scores[name]["distance"]) for name in SERIES},
        **geometry,
    }
    if any(
        value is None or (isinstance(value, float) and not math.isfinite(value))
        for value in record.values()
    ):
        # 欠測・NaN・Inf を学習データへ流さない（不変条件 11）。0 で埋めない。
        return ("non_finite_feature", None)

    coords = None
    if RETURN_LANDMARK_COORDS:
        coords = normalized.astype(np.float32).reshape(-1)  # 478*3 = 1434
        if not np.isfinite(coords).all():
            return ("non_finite_feature", None)
    return ("ok", (record, coords))


In [ ]:
import sys
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp_ctx

import numpy as np

sys.path.insert(0, '/content')   # spawn した子プロセスが import できるように
import fsl_age_gate

# --- 母集団：ファイル名注釈が読める画像すべて（年齢で絞らない）---
inventory_exclusions = Counter()
candidates = []
for path in image_paths:
    annotation = fsl_age_gate.parse_utkface_name(path.name)
    if annotation is None:
        inventory_exclusions['unparsable_filename'] += 1
        continue
    candidates.append((path, annotation))

assert candidates, '注釈を読める画像が 0 件でした。'

ages_all = np.array([a['age'] for _, a in candidates])
print(f'candidates: {len(candidates):,} / {len(image_paths):,}')
print('age range :', int(ages_all.min()), '-', int(ages_all.max()))
print('in-range  :', int(((ages_all >= IN_RANGE_MIN_AGE) & (ages_all <= IN_RANGE_MAX_AGE)).sum()))
print('under 18  :', int((ages_all < IN_RANGE_MIN_AGE).sum()))
print('over 75   :', int((ages_all > IN_RANGE_MAX_AGE).sum()))
print('inventory exclusions:', dict(inventory_exclusions))

if MAX_IMAGES is None or MAX_IMAGES >= len(candidates):
    selected = tuple(candidates)
else:
    generator = np.random.default_rng(RANDOM_SEED)
    indices = sorted(
        int(i) for i in generator.choice(len(candidates), size=MAX_IMAGES, replace=False)
    )
    selected = tuple(candidates[i] for i in indices)

path_strings = [str(path) for path, _ in selected]
gates = (MAX_ABS_YAW_DEG, MAX_ABS_PITCH_DEG, MAX_ABS_ROLL_DEG)

records = []            # 1 要素 = 1 画像の個票（画像そのものは持たない）
coord_rows = []         # 正規化座標 1434 次元（メモリ上だけ。保存しない）
analysis_exclusions = Counter()


def _consume(image_id, path, annotation, status, payload):
    """測定できた画像だけを積む。失敗は理由別に数えるだけで止めない（不変条件12）。"""
    if status != 'ok':
        analysis_exclusions[status] += 1
        return
    record, coords = payload
    records.append({'image_id': image_id, 'filename': path.name, **annotation, **record})
    if INCLUDE_LANDMARK_COORDS:
        coord_rows.append(coords)


if USE_MULTIPROCESSING and N_WORKERS > 1:
    context = mp_ctx.get_context('spawn')     # MediaPipe を fork しない
    with ProcessPoolExecutor(
        max_workers=N_WORKERS,
        mp_context=context,
        initializer=fsl_age_gate.init_worker,
        initargs=(str(MODEL_PATH), gates, INCLUDE_LANDMARK_COORDS),
    ) as executor:
        outcomes = executor.map(
            fsl_age_gate.analyze_one_path, path_strings, chunksize=CHUNK
        )
        for image_id, ((path, annotation), (status, payload)) in enumerate(
            zip(selected, outcomes), start=1
        ):
            _consume(image_id, path, annotation, status, payload)
            if image_id % PROGRESS_EVERY == 0 or image_id == len(selected):
                print(f'processed {image_id:,}/{len(selected):,} (valid={len(records):,})')
else:
    fsl_age_gate.init_worker(str(MODEL_PATH), gates, INCLUDE_LANDMARK_COORDS)
    try:
        for image_id, (path, annotation) in enumerate(selected, start=1):
            status, payload = fsl_age_gate.analyze_one_path(str(path))
            _consume(image_id, path, annotation, status, payload)
            if image_id % PROGRESS_EVERY == 0 or image_id == len(selected):
                print(f'processed {image_id:,}/{len(selected):,} (valid={len(records):,})')
    finally:
        if fsl_age_gate._LANDMARKER is not None:
            fsl_age_gate._LANDMARKER.close()

assert records, '姿勢ゲートと特徴抽出を通過した画像が 0 件でした。'
LANDMARK_MATRIX = (
    np.vstack(coord_rows).astype(np.float32) if INCLUDE_LANDMARK_COORDS else None
)
coord_rows = None        # 参照を切ってメモリを返す

print(f'valid: {len(records):,}/{len(selected):,}')
print('analysis exclusions:', dict(analysis_exclusions))
if LANDMARK_MATRIX is not None:
    print('landmark matrix   :', LANDMARK_MATRIX.shape, LANDMARK_MATRIX.dtype)

In [ ]:
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

frame = pd.DataFrame.from_records(records).sort_values('image_id').reset_index(drop=True)

# ---- ラベル：1 = 範囲内（18〜75）、0 = 範囲外 ----
frame['in_range'] = (
    (frame['age'] >= IN_RANGE_MIN_AGE) & (frame['age'] <= IN_RANGE_MAX_AGE)
).astype(int)
frame['age_group'] = np.where(
    frame['age'] < IN_RANGE_MIN_AGE, 'under_18',
    np.where(frame['age'] > IN_RANGE_MAX_AGE, 'over_75', 'in_range'),
)

print(frame['age_group'].value_counts().to_dict())
print('陽性率（範囲内）: %.4f' % frame['in_range'].mean())

# ---- 分割：層は「年齢グループ × 10歳きざみの年齢」。境界付近を全 split へ行き渡らせる ----
# 同一人物が複数枚含まれている可能性は UTKFace では検証できない。
# 個人 ID が無いため identity leakage は排除できない、と評価レポートへ明記する。
strata = (
    frame['age_group'].astype(str) + '_' + (frame['age'] // 10).astype(str)
)
counts = strata.value_counts()
strata = strata.where(strata.map(counts) >= 6, other='rare')   # 極小の層はまとめる

from sklearn.model_selection import train_test_split

indices = np.arange(len(frame))
train_idx, temp_idx = train_test_split(
    indices, test_size=SPLIT_RATIOS[1] + SPLIT_RATIOS[2],
    random_state=RANDOM_SEED, stratify=strata,
)
val_ratio = SPLIT_RATIOS[2] / (SPLIT_RATIOS[1] + SPLIT_RATIOS[2])
val_idx, test_idx = train_test_split(
    temp_idx, test_size=val_ratio, random_state=RANDOM_SEED,
    stratify=strata.iloc[temp_idx],
)
frame['split'] = 'train'
frame.loc[val_idx, 'split'] = 'validation'
frame.loc[test_idx, 'split'] = 'test'

print(frame.groupby(['split', 'age_group']).size().unstack(fill_value=0))

# ---- 特徴行列を 2 種類そろえる ----
TABULAR_NAMES = list(fsl_age_gate.TABULAR_FEATURE_NAMES)
X_ratio = frame[TABULAR_NAMES].to_numpy(dtype=np.float32)
assert np.isfinite(X_ratio).all(), '特徴量に NaN / Inf が混入しています'

FEATURE_SETS = {'ratio': (TABULAR_NAMES, X_ratio)}
if LANDMARK_MATRIX is not None:
    coord_names = [
        f'lm{index:03d}_{axis}' for index in range(478) for axis in ('x', 'y', 'z')
    ]
    FEATURE_SETS['landmark'] = (
        TABULAR_NAMES + coord_names,
        np.hstack([X_ratio, LANDMARK_MATRIX]).astype(np.float32),
    )

for name, (names, matrix) in FEATURE_SETS.items():
    print(f'{name:9s}: {matrix.shape[1]:5d} 次元')
display(frame[['age', 'age_group', 'split', *TABULAR_NAMES[:6]]].head())

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

y = frame['in_range'].to_numpy(dtype=np.int64)
ages = frame['age'].to_numpy(dtype=np.int64)
is_train = (frame['split'] == 'train').to_numpy()
is_val = (frame['split'] == 'validation').to_numpy()
is_test = (frame['split'] == 'test').to_numpy()

# ---- ベースライン：常に「範囲内」と答える ----
# ここに勝てなければ学習した意味がない。特に under_18 の再現率は 0 になる。
baseline_accuracy = float(y[is_test].mean())
print(f'ベースライン（常に範囲内）: accuracy {baseline_accuracy:.4f} / under18 recall 0.0000')

LGBM_PARAMS = dict(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    class_weight='balanced',   # 範囲外が少数派なので重みで補正する
    random_state=RANDOM_SEED,
    n_jobs=N_WORKERS,
    verbose=-1,
)

trained = {}
for name, (names, matrix) in FEATURE_SETS.items():
    model = LGBMClassifier(**LGBM_PARAMS)
    model.fit(matrix[is_train], y[is_train])
    proba_val = model.predict_proba(matrix[is_val])[:, 1]     # 範囲内である確率
    trained[name] = {
        'model': model,
        'feature_names': names,
        'matrix': matrix,
        'proba_val': proba_val,
        'pr_auc': float(average_precision_score(y[is_val], proba_val)),
        'roc_auc': float(roc_auc_score(y[is_val], proba_val)),
    }
    print(f'{name:9s}: val PR-AUC {trained[name]["pr_auc"]:.4f}  '
          f'ROC-AUC {trained[name]["roc_auc"]:.4f}')

best_name = max(trained, key=lambda key: trained[key]['pr_auc'])
print(f'\n検証で良かった特徴セット: {best_name}')
print('（書き出す方は EXPORT_FEATURE_SET で選ぶ。比率特徴は次元が小さくアプリ側の実装も軽い）')

In [ ]:
# ---- 動作点の決定 ----
# p = 範囲内である確率。p <= threshold のとき「範囲外」と判定する。
# 迷ったら範囲外へ倒したいので、「18歳未満をどれだけ確実に弾けるか」を先に決め、
# その条件を満たすなかで「範囲内をどれだけ残せるか」が最大の閾値を選ぶ。

def choose_threshold(proba, ages_subset, target_recall):
    minors = ages_subset < IN_RANGE_MIN_AGE
    in_range = (ages_subset >= IN_RANGE_MIN_AGE) & (ages_subset <= IN_RANGE_MAX_AGE)
    best = None
    for threshold in np.unique(np.round(np.append(proba, [0.0, 1.0]), 4)):
        recall = float((proba[minors] <= threshold).mean()) if minors.any() else 1.0
        if recall < target_recall:
            continue
        kept = float((proba[in_range] > threshold).mean())
        if best is None or kept > best['in_range_kept_val']:
            best = {
                'threshold': float(threshold),
                'under18_recall_val': recall,
                'in_range_kept_val': kept,
            }
    if best is None:
        raise ValueError(f'目標再現率 {target_recall} を満たす閾値がありません')
    return best


def evaluate(proba, ages_subset, threshold):
    """閾値つきの判定を、隠さずに分解して出す。"""
    predicted_in = proba > threshold
    truth_in = (ages_subset >= IN_RANGE_MIN_AGE) & (ages_subset <= IN_RANGE_MAX_AGE)
    minors = ages_subset < IN_RANGE_MIN_AGE
    seniors = ages_subset > IN_RANGE_MAX_AGE
    return {
        'threshold': float(threshold),
        'count': int(len(ages_subset)),
        'accuracy': float((predicted_in == truth_in).mean()),
        'true_positive': int((predicted_in & truth_in).sum()),
        'false_negative': int((~predicted_in & truth_in).sum()),
        'true_negative': int((~predicted_in & ~truth_in).sum()),
        'false_positive': int((predicted_in & ~truth_in).sum()),
        'in_range_kept': float(predicted_in[truth_in].mean()),
        'under18_recall': float((~predicted_in[minors]).mean()) if minors.any() else None,
        'over75_recall': float((~predicted_in[seniors]).mean()) if seniors.any() else None,
    }


selected_set = EXPORT_FEATURE_SET
bundle = trained[selected_set]
matrix = bundle['matrix']
bundle['proba_test'] = bundle['model'].predict_proba(matrix[is_test])[:, 1]

operating_points = {}
for target in TARGET_UNDER18_RECALLS:
    chosen = choose_threshold(bundle['proba_val'], ages[is_val], target)
    result = evaluate(bundle['proba_test'], ages[is_test], chosen['threshold'])
    operating_points[f'{target:.2f}'] = {**chosen, 'test': result}
    print(
        f'目標 under18 recall {target:.2f} → 閾値 {chosen["threshold"]:.4f} | '
        f'test: under18 {result["under18_recall"]:.3f}  '
        f'over75 {result["over75_recall"]:.3f}  '
        f'範囲内を残せた割合 {result["in_range_kept"]:.3f}'
    )

PRIMARY = operating_points[f'{PRIMARY_TARGET_UNDER18_RECALL:.2f}']
THRESHOLD = PRIMARY['threshold']
print(f'\n採用する閾値: {THRESHOLD:.4f}')
print('※ 18歳未満と76歳以上では難易度が違う。まとめた精度ではなく必ず分けて読むこと。')

In [ ]:
import matplotlib.pyplot as plt

frame['proba_in_range'] = np.nan
for mask, key in ((is_val, 'proba_val'), (is_test, 'proba_test')):
    frame.loc[mask, 'proba_in_range'] = bundle[key]
frame.loc[is_train, 'proba_in_range'] = (
    bundle['model'].predict_proba(matrix[is_train])[:, 1]
)
frame['predicted_in_range'] = (frame['proba_in_range'] > THRESHOLD).astype(int)

test_frame = frame[is_test]

# ---- 1) 年齢ごとの「範囲外と正しく言えた率」。境界がぼやけることを隠さない ----
bins = [(0, 5), (6, 11), (12, 14), (15, 17), (18, 20), (21, 30), (31, 50),
        (51, 70), (71, 75), (76, 85), (86, 120)]
rows = []
for low, high in bins:
    subset = test_frame[(test_frame['age'] >= low) & (test_frame['age'] <= high)]
    if len(subset) < MIN_GROUP_FOR_REPORT:
        continue
    truth_in = int(low >= IN_RANGE_MIN_AGE and high <= IN_RANGE_MAX_AGE)
    rows.append({
        'age_bin': f'{low}-{high}',
        'count': int(len(subset)),
        'truth': 'in_range' if truth_in else 'out_of_range',
        'correct_rate': float((subset['predicted_in_range'] == truth_in).mean()),
        'mean_proba': float(subset['proba_in_range'].mean()),
    })
age_report = pd.DataFrame(rows)
display(age_report)

# ---- 2) 群ごとの誤り率。偏りを隠したまま配備しない ----
group_rows = []
for dimension in ('race', 'gender'):
    for value, subset in test_frame.groupby(dimension):
        for label, part in (('in_range', subset[subset['in_range'] == 1]),
                            ('out_of_range', subset[subset['in_range'] == 0])):
            if len(part) < MIN_GROUP_FOR_REPORT:
                continue
            correct = int(label == 'in_range')
            group_rows.append({
                'dimension': dimension, 'group': str(value), 'truth': label,
                'count': int(len(part)),
                'error_rate': float((part['predicted_in_range'] != correct).mean()),
            })
group_report = pd.DataFrame(group_rows)
display(group_report)

# ---- 3) 図：年齢ごとの予測確率の分布 ----
# 図のラベルは英語にする。Colab の既定フォントは日本語グリフを持たず豆腐になるため。
figure, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(test_frame['age'], test_frame['proba_in_range'], s=4, alpha=0.25)
axes[0].axhline(THRESHOLD, color='crimson', linewidth=1.2, label=f'threshold {THRESHOLD:.3f}')
for edge in (IN_RANGE_MIN_AGE, IN_RANGE_MAX_AGE):
    axes[0].axvline(edge, color='gray', linestyle='--', linewidth=1)
axes[0].set_xlabel('age (dataset annotation)')
axes[0].set_ylabel('P(in range)')
axes[0].set_title('P(in range) by annotated age — test split')
axes[0].legend()

if not age_report.empty:
    axes[1].bar(age_report['age_bin'], age_report['correct_rate'])
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel('correct rate')
    axes[1].set_title('correct rate by age bin — test split')
    axes[1].tick_params(axis='x', rotation=60)
figure.tight_layout()
plt.show()

# ---- 4) 特徴量の寄与（比率特徴のときだけ意味を読める）----
importances = pd.Series(
    bundle['model'].feature_importances_, index=bundle['feature_names']
).sort_values(ascending=False)
display(importances.head(15).to_frame('importance'))

In [ ]:
import importlib.metadata
import json
import platform
from datetime import UTC, datetime

import onnxruntime as ort
from onnxmltools.convert import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

OUTPUT_DIR = Path('/content')
ONNX_PATH = OUTPUT_DIR / 'age_gate_model.onnx'
SPEC_PATH = OUTPUT_DIR / 'age_gate_feature_spec.json'
CARD_PATH = OUTPUT_DIR / 'age_gate_model_card.md'
EVAL_PATH = OUTPUT_DIR / 'utkface_agegate_eval.json'
SUMMARY_PATH = OUTPUT_DIR / 'utkface_agegate_summary.json'
RECORDS_PATH = OUTPUT_DIR / 'utkface_agegate_records.csv'

GATE_VERSION = 'age-gate/1.0.0'


def _package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return 'not installed'


def json_default(value):
    """numpy 由来の型を素の Python へ落とす。json.dumps はこれが無いと落ちる。"""
    if hasattr(value, 'item'):
        return value.item()
    return str(value)
TARGET_OPSET = 15    # onnxmltools 1.16 の LightGBM converter の上限

feature_names = bundle['feature_names']
onnx_model = convert_lightgbm(
    bundle['model'],
    initial_types=[('features', FloatTensorType([None, len(feature_names)]))],
    target_opset=TARGET_OPSET,
    zipmap=False,
)
ONNX_PATH.write_bytes(onnx_model.SerializeToString())

# ---- パリティ検証：ONNX と学習器が同じ値を返すことを確かめる ----
session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
onnx_probability = session.run(None, {'features': matrix[is_test]})[1][:, 1]
parity = float(np.abs(onnx_probability - bundle['proba_test']).max())
assert parity < 1e-5, f'ONNX と学習器の出力が一致しません: {parity}'
print(f'ONNX 書き出し完了: {ONNX_PATH.name}  parity max|diff| = {parity:.3e}')

ONNX_SHA256 = sha256_file(ONNX_PATH)

# ---- 特徴量仕様：アプリはこの順序でベクトルを組み立てる ----
SPEC_PATH.write_text(json.dumps({
    'feature_spec_version': fsl_age_gate.FEATURE_SPEC_VERSION,
    'gate_version': GATE_VERSION,
    'feature_set': selected_set,
    'input_name': 'features',
    'input_dtype': 'float32',
    'feature_names': feature_names,
    'normalization': {
        'function': 'normalize_landmarks_3d',
        'description': '両眼中点を原点、3Dの両眼間距離を1、両眼を水平へ回した座標',
        'landmark_map_version': fsl_age_gate.LANDMARK_MAP_VERSION,
        'landmark_count': fsl_age_gate.LANDMARK_COUNT,
    },
    'pose_gate_deg': {
        'max_abs_yaw': MAX_ABS_YAW_DEG,
        'max_abs_pitch': MAX_ABS_PITCH_DEG,
        'max_abs_roll': MAX_ABS_ROLL_DEG,
    },
    'output': {
        'probabilities_index_in_range': 1,
        'decision': 'probabilities[1] > threshold のとき「範囲内」。それ以外は「範囲外」',
        'threshold': THRESHOLD,
    },
}, ensure_ascii=False, indent=2, default=json_default), encoding='utf-8')

# ---- 固定評価レポート ----
evaluation = {
    'gate_version': GATE_VERSION,
    'generated_at': datetime.now(UTC).isoformat(),
    'feature_set': selected_set,
    'feature_count': len(feature_names),
    'split_counts': frame['split'].value_counts().to_dict(),
    'baseline_always_in_range_accuracy': baseline_accuracy,
    'validation': {
        name: {'pr_auc': trained[name]['pr_auc'], 'roc_auc': trained[name]['roc_auc']}
        for name in trained
    },
    'operating_points': operating_points,
    'primary_operating_point': PRIMARY,
    'age_bin_report': age_report.to_dict(orient='records'),
    'group_error_report': group_report.to_dict(orient='records'),
    'known_limitations': [
        '17歳と18歳、75歳と76歳は原理的に分離できない。境界付近の誤りは設計上の前提である。',
        '18歳未満と76歳以上は難易度が大きく違う。まとめた精度で読んではならない。',
        'UTKFaceの年齢注釈自体の誤りは検証していない。天井精度は注釈ノイズで決まる。',
        'UTKFaceには個人IDが無いため、同一人物がsplitをまたぐ可能性を排除できていない。',
        '姿勢ゲート（yaw/pitch/roll）を通ったフレームでしか評価していない。',
        '群ごとの誤り率に差がある場合、それは配備時の実害になりうる。group_error_reportを必ず読む。',
    ],
}
EVAL_PATH.write_text(json.dumps(evaluation, ensure_ascii=False, indent=2, default=json_default), encoding='utf-8')

# ---- 実行条件のサマリ ----
summary = {
    'schema_version': 'utkface-age-gate/1.0.0',
    'gate_version': GATE_VERSION,
    'status': 'research_prototype_not_an_age_verification_system',
    'generated_at': datetime.now(UTC).isoformat(),
    'purpose': (
        'アプリが参考年齢を表示してよいかどうかだけを判定するゲート。'
        '年齢認証・本人確認・未成年判定・入場制限には使用できない。'
    ),
    'dataset': {
        'name': 'UTKFace',
        'variant': 'Aligned & Cropped Faces',
        'official_page': UTKFACE_OFFICIAL_PAGE_URL,
        'download_mirror': DATASET_MIRROR_URL,
        'download_mirror_handle': DATASET_MIRROR_HANDLE,
        'license_scope': 'non-commercial research only',
        'inventory_sha256_names_and_sizes': DATASET_INVENTORY_SHA256,
        'label_source': 'filename annotation: age_gender_race_datetime.jpg',
        'label_caveat': '年齢はUTKFaceの配布注釈であり、本notebookは推定しない。注釈自体の誤りは検証していない。',
    },
    'protocol': {
        'in_range': [IN_RANGE_MIN_AGE, IN_RANGE_MAX_AGE],
        'age_gate': '掛けない（範囲外の画像が負例）',
        'max_abs_yaw_deg': MAX_ABS_YAW_DEG,
        'max_abs_pitch_deg': MAX_ABS_PITCH_DEG,
        'max_abs_roll_deg': MAX_ABS_ROLL_DEG,
        'random_seed': RANDOM_SEED,
        'split_ratios': list(SPLIT_RATIOS),
        'stratification': 'age_group × 10歳きざみの年齢',
    },
    'counts': {
        'discovered_images': len(image_paths),
        'candidate_images': len(candidates),
        'selected_images': len(selected),
        'valid_images': len(frame),
        'inventory_exclusions': dict(inventory_exclusions),
        'analysis_exclusions': dict(analysis_exclusions),
        'age_group_totals': frame['age_group'].value_counts().to_dict(),
    },
    'reproducibility': {
        'metric_version': fsl_age_gate.METRIC_VERSION,
        'landmark_map_version': fsl_age_gate.LANDMARK_MAP_VERSION,
        'feature_spec_version': fsl_age_gate.FEATURE_SPEC_VERSION,
        'landmark_model_version': MODEL_VERSION,
        'landmark_model_sha256': MODEL_SHA256,
        'landmark_delegate': 'CPU',
        'onnx_sha256': ONNX_SHA256,
        'onnx_target_opset': TARGET_OPSET,
        'lgbm_params': {key: value for key, value in LGBM_PARAMS.items()},
        'python': platform.python_version(),
        'packages': {
            name: _package_version(name)
            for name in ('mediapipe', 'numpy', 'pandas', 'pillow',
                         'opencv-contrib-python', 'lightgbm', 'onnx',
                         'onnxmltools', 'onnxruntime', 'scikit-learn')
        },
    },
    'privacy': {
        'contains_images': False,
        'contains_landmarks': SAVE_LANDMARK_MATRIX,
        'contains_per_image_predictions': True,
        'contains_filenames': True,
        'note': '個票は後続分析のために保存する。画像・ランドマーク座標は保存しない。',
    },
    'warning': (
        'このゲートは「参考年齢を表示してよいか」だけを判定する研究用プロトタイプである。'
        '年齢認証・本人確認・未成年判定・入場制限・その他の権利や機会に影響する判断へ使用してはならない。'
        '境界年齢（17/18歳、75/76歳）は原理的に分離できず、群ごとに誤り率が異なる。'
        'UTKFaceと固定した選別条件に限定した結果であり、他の撮影条件・他の集団へは外挿できない。'
    ),
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=json_default), encoding='utf-8')

# ---- モデルカード ----
CARD_PATH.write_text(f"""# Model card — {GATE_VERSION}

## 用途

18〜75歳の範囲内かどうかだけを判定し、**アプリが参考年齢を表示してよいかを決める**。

## 使ってはならない用途

年齢認証、本人確認、未成年判定、入場制限、採用、保険、教育、監視、人物間比較。

## 入力

- `features`: float32 `[N, {len(feature_names)}]`。順序は `age_gate_feature_spec.json` の `feature_names`。
- 前段の条件: 単一顔であること、姿勢ゲート（|yaw|<={MAX_ABS_YAW_DEG}°, |pitch|<={MAX_ABS_PITCH_DEG}°, |roll|<={MAX_ABS_ROLL_DEG}°）を通っていること。

## 出力

- `probabilities[:, 1]` = 範囲内である確率。
- `probabilities[:, 1] > {THRESHOLD:.4f}` のとき「範囲内」。それ以外は「範囲外」。
- 閾値は**18歳未満の再現率 {PRIMARY_TARGET_UNDER18_RECALL:.2f} 以上**を満たす動作点として検証データで選んだ。迷ったら範囲外へ倒す設計である。

## 学習データ

UTKFace Aligned & Cropped（非商用研究限定）。年齢・性別・人種はデータセットの配布注釈であり推定していない。
inventory SHA-256: `{DATASET_INVENTORY_SHA256}`

## 評価（固定テストセット）

| 指標 | 値 |
|---|---|
| ベースライン（常に範囲内）accuracy | {baseline_accuracy:.4f} |
| accuracy | {PRIMARY['test']['accuracy']:.4f} |
| 18歳未満を範囲外と言えた率 | {PRIMARY['test']['under18_recall']:.4f} |
| 76歳以上を範囲外と言えた率 | {PRIMARY['test']['over75_recall']:.4f} |
| 範囲内を範囲内と言えた率 | {PRIMARY['test']['in_range_kept']:.4f} |

群ごとの誤り率は `utkface_agegate_eval.json` の `group_error_report` を参照する。

## 限界

- 17歳と18歳、75歳と76歳は原理的に分離できない。
- 18歳未満と76歳以上では難易度が大きく違う。まとめた精度で読まない。
- UTKFace の年齢注釈自体に誤りが含まれる。天井精度は注釈ノイズで決まる。
- 個人 ID が無いため、同一人物が split をまたぐ可能性を排除できていない。
- 姿勢ゲートを通ったフレームでしか評価していない。

## ハッシュ

- ONNX SHA-256: `{ONNX_SHA256}`
- landmark model SHA-256: `{MODEL_SHA256}`
""", encoding='utf-8')

# ---- 個票（ランドマーク座標は含まない）----
frame.to_csv(RECORDS_PATH, index=False, encoding='utf-8')

for path in (ONNX_PATH, SPEC_PATH, CARD_PATH, EVAL_PATH, SUMMARY_PATH, RECORDS_PATH):
    print(f'{path.name:34s} {path.stat().st_size:>10,} bytes')

print('\n' + summary['warning'])

In [ ]:
from google.colab import files

for path in (ONNX_PATH, SPEC_PATH, CARD_PATH, EVAL_PATH, SUMMARY_PATH, RECORDS_PATH):
    files.download(str(path))

## アプリへの組み込み手順（face-symmetry-lab 側）

このnotebookが渡すのは **ONNX 1本 + 仕様JSON + モデルカード + 固定評価レポート**だけである。
学習コードとcheckpointをアプリから参照しない、という既存の境界をそのまま守る。

1. `models/` へ `age_gate_model.onnx` を置き、`configs/model-manifest.json` へ
   URL・サイズ・SHA-256 を追記する（`adapters/io/model_store.py` の hash 検証に載せる）。
2. `docs/model-cards/age-gate-v1.md` として `age_gate_model_card.md` を追加する。
3. `domain/` に特徴量の組み立てを純粋関数として実装する。
   `age_gate_feature_spec.json` の `feature_names` と**同じ順序**で並べること。
   既存の `domain/geometry.py::normalize_landmarks_3d` をそのまま使えるので、
   足すのは比率の計算だけになる。
4. `adapters/inference/` へ ONNX Runtime のアダプタを足し、`application/` の
   port 経由で呼ぶ。**姿勢ゲートを通り単一顔が取れたときだけ**呼ぶ。
5. ゲートが「範囲外」と言ったら、**参考年齢の区画だけ**を「対象範囲外」と表示する。
   左右差の分析・表示・エクスポートは止めない（不変条件12の失敗分離）。
6. `requirement.md` へ新しい要件IDを立て、§10 へ「年齢認証には使わない」旨と
   群ごとの誤り率の開示を書く。テストは実装より先に書く。

### 特徴量の一致を必ずテストする

学習時とアプリ側で特徴量の並びが1つでもずれると、モデルは黙って間違った値を返す。
**notebookが出した数枚分の特徴ベクトルをfixtureとして固定し、アプリ側の実装が
同じ値を返すことをcontract testで確認すること。** これが最も壊れやすい箇所である。
